# 54 · Cost controls in production
Level L5 · Part 6 · professional · ~30 min · builds on 12, 30, 33 (also uses 07, 14, 53)

## Goal
Give Scout a hard ceiling on spend: $0.01 per run and a daily limit per user, enforced *before* each model
call. Send easy questions to the small model first. Stop cleanly when the provider's spend cap answers
with the `enforced_spend_limit_reached` 429.

## The idea
An agent decides for itself how many model calls to make, so its cost per run is not fixed. Four controls
bound it. **`max_output_tokens`** caps one call. A **budget** caps a run (and a user's day): before each
call, the loop checks that spend so far plus the next call's worst case still fits.
**Model routing** sends easy requests to a cheap model and only hard ones to a dear one. A **provider
spend limit** is the last line, outside your code: when it is hit, every call fails until the limit resets.

Analogy: a prepaid travel card. Each purchase is checked against the balance before it goes through, the
daily limit is separate from the card's total, and the bank can freeze the card whatever your app thinks.

## Picture
```mermaid
flowchart LR
    Q["question"] --> D{"user's daily<br/>budget left?"}
    D -- no --> SD["stop: daily_budget"]
    D -- yes --> R{"router"}
    R -- easy --> S["small model"]
    R -- hard --> B["big model"]
    S & B --> C{"spent + worst case<br/>of next call <= run budget?"}
    C -- no --> SB["stop: budget"]
    C -- yes --> M["model call"] --> P{"provider"}
    P -- "429 spend cap" --> SC["stop: spend_cap, alert"]
    P -- ok --> CH["charge usage"] --> C
```

## Step by step
### 1. Start the recorder and set up
The recorder first. Without a key the Gemini calls replay from
`../data/llm_cassettes/54_cost-controls.yaml`.

In [1]:
import sys; sys.path.insert(0, "..")
import llm_replay
llm_replay.start("54_cost-controls")

LLM mode: replay (54_cost-controls.yaml)


Setup from earlier lessons, unchanged: the notes, client and pinned model (lesson 07), the dated price
table and `cost_usd` (lesson 12).

In [2]:
import ast, json, operator, os
from dotenv import load_dotenv
from google import genai
from google.genai import types

NOTES = [
    {"id": 1, "text": "Mars has two small moons, Phobos and Deimos."},
    {"id": 2, "text": "Venus spins backwards compared with most planets."},
    {"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes."},
    {"id": 4, "text": "Jupiter is the largest planet in the solar system."},
    {"id": 5, "text": "The rover Perseverance landed on Mars in February 2021."},
    {"id": 6, "text": "Saturn's rings are mostly made of ice."},
]
load_dotenv("../.env")
client = genai.Client(vertexai=True, api_key=os.environ.get("GOOGLE_CLOUD_API_KEY") or "replay-no-key-needed")
PRICES_PER_MTOK = {            # USD per 1,000,000 tokens; source: Gemini pricing page, 2026-10-06
    "gemini-3.5-flash-lite": {"input": 0.30, "output": 2.50},
    "gemini-3.8-flash":      {"input": 0.75, "output": 3.75},   # promo price until 2026-12-31, then 1.50 / 7.50
}

def cost_usd(model: str, input_tokens: int, output_tokens: int) -> float:
    price = PRICES_PER_MTOK[model]
    return (input_tokens * price["input"] + output_tokens * price["output"]) / 1_000_000

Scout's two tools, their schemas and `run_tool` from lesson 30, unchanged.

In [3]:
def search_notes(query: str) -> list[dict]:
    """Return the notes whose text contains `query` (ignoring case)."""
    return [note for note in NOTES if query.lower() in note["text"].lower()]

OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}

def calculator(expression: str) -> float:
    """Evaluate +, -, *, / on numbers. Nothing else is allowed."""
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in OPS:
            return OPS[type(node.op)](ev(node.left), ev(node.right))
        raise ValueError(f"not allowed in a calculation: {ast.dump(node)[:40]}")
    return ev(ast.parse(expression, mode="eval").body)

def run_tool(call: dict, tools: dict) -> dict:
    fn = tools.get(call["name"])
    if fn is None:
        return {"content": f"unknown tool '{call['name']}'", "is_error": True}
    try:
        return {"content": json.dumps(fn(**call["args"])), "is_error": False}
    except Exception as e:
        return {"content": f"{type(e).__name__}: {e}", "is_error": True}

The schemas, as in lesson 30.

In [4]:
SCHEMAS = [
    {"name": "search_notes", "description": "Search the user's notes. Returns every note whose text contains "
     "the query word (case-insensitive), as a list of {id, text}. Use it before answering any question "
     "about what the notes say.", "parameters": {"type": "object", "required": ["query"], "properties":
     {"query": {"type": "string", "description": "one word to look for, e.g. 'Mars'"}}}},
    {"name": "calculator", "description": "Evaluate an arithmetic expression with numbers, + - * / and "
     "brackets, e.g. '24*60+39'. Use it for every calculation instead of doing arithmetic yourself.",
     "parameters": {"type": "object", "required": ["expression"],
                    "properties": {"expression": {"type": "string", "description": "e.g. '(3+4)*2'"}}}},
]
TOOLS = {"search_notes": search_notes, "calculator": calculator}
print(len(NOTES), "notes |", [s["name"] for s in SCHEMAS], "|", calculator("24*60+39"))

6 notes | ['search_notes', 'calculator'] | 1479


### 2. Control 1: `max_output_tokens` bounds one call
Lesson 12 showed the worst case of a call: the exact input cost plus the output cap at the output price.
The cap is what makes a worst case exist at all. Here is one call with 500 input tokens on each model,
at three caps. Thinking counts toward the cap and is billed as output (lesson 40).

In [5]:
for model in PRICES_PER_MTOK:
    worst = {cap: cost_usd(model, 500, cap) for cap in (300, 1024, 8192)}
    print(f"{model:22}", " | ".join(f"cap {cap:>5}: ${usd:.5f}" for cap, usd in worst.items()))

gemini-3.5-flash-lite  cap   300: $0.00090 | cap  1024: $0.00271 | cap  8192: $0.02063
gemini-3.8-flash       cap   300: $0.00150 | cap  1024: $0.00421 | cap  8192: $0.03110


On the big model, a call capped at 8,192 tokens can cost $0.031, three times a $0.01 run budget, in one
call. A cap of 300 keeps the small model's worst call under a tenth of a cent. Pick the cap per model
from what its answers actually need, not "as high as it goes".

### 3. Control 2: a per-run budget
A **run budget** limits the money and tokens one agent run may use. The key rule: check *before* the call,
with the worst case of that call, so the run can never go over. We do not know the next call's input size
exactly, so we estimate it as the last call's input plus a margin for what was added since (a tool result
or two). Lesson 12's `count_tokens` gives the exact number, for one extra request per step.

In [6]:
class RunBudget:
    def __init__(self, max_usd: float, max_tokens: int = 50_000, first_input: int = 600, margin: int = 200):
        self.max_usd, self.max_tokens, self.margin = max_usd, max_tokens, margin
        self.spent_usd, self.tokens, self.last_in = 0.0, 0, first_input - margin

    def worst_next(self, model: str, max_out: int) -> tuple[float, int]:
        est_in = self.last_in + self.margin
        return cost_usd(model, est_in, max_out), est_in + max_out

    def can_afford(self, model: str, max_out: int) -> bool:
        usd, tok = self.worst_next(model, max_out)
        return self.spent_usd + usd <= self.max_usd and self.tokens + tok <= self.max_tokens

    def charge(self, model: str, tokens_in: int, tokens_out: int) -> None:
        self.spent_usd += cost_usd(model, tokens_in, tokens_out)
        self.tokens += tokens_in + tokens_out
        self.last_in = tokens_in

print(RunBudget(0.01).worst_next("gemini-3.8-flash", 1024))

(0.00429, 1624)


The smallest example: a $0.01 budget, three calls of 1,500 in and 250 out on the big model, then ask
again before each next call.

In [7]:
b = RunBudget(max_usd=0.01)
for i in range(1, 4):
    print(f"before call {i}: can afford = {b.can_afford('gemini-3.8-flash', 1024)}", end=" | ")
    b.charge("gemini-3.8-flash", 1500, 250)
    print(f"spent ${b.spent_usd:.5f}, {b.tokens} tokens")
print("worst case of call 4:", f"${b.worst_next('gemini-3.8-flash', 1024)[0]:.5f}",
      "| can afford:", b.can_afford("gemini-3.8-flash", 1024))

before call 1: can afford = True | spent $0.00206, 1750 tokens
before call 2: can afford = True | spent $0.00413, 3500 tokens
before call 3: can afford = True | spent $0.00619, 5250 tokens
worst case of call 4: $0.00511 | can afford: False


After three calls only $0.0062 is spent, yet call 4 is refused: its worst case ($0.0051) would take the
run past $0.01. The budget holds back room for the *worst* call, not the average one.

### 4. The budget inside the agent loop
Lesson 30's loop, cut down to the step limit plus the new guard. Each model now reports its `name`, its
`max_out` cap and the `usage` of each reply, so the loop can check before the call and charge after it.

In [8]:
def run_agent(question: str, model, budget: RunBudget, max_steps: int = 8) -> dict:
    messages = [{"role": "user", "text": question}]
    def finish(stop, steps, answer=None):
        return dict(answer=answer, stop=stop, steps=steps, model=model.name, spent_usd=round(budget.spent_usd, 6),
                    tokens=budget.tokens, messages=messages)
    for step in range(1, max_steps + 1):
        if not budget.can_afford(model.name, model.max_out):        # new: check before paying
            return finish("budget", step - 1)
        reply = model(messages, SCHEMAS)
        budget.charge(model.name, *reply.pop("usage"))              # new: pay for what was used
        messages.append(reply)
        calls = reply.get("tool_calls", [])
        if not calls:
            return finish("end_of_turn", step, reply["text"])
        for call in calls:
            messages.append({"role": "tool", "call_id": call["id"], "name": call["name"], **run_tool(call, TOOLS)})
    return finish("step_limit", max_steps)

A scripted model with a price tag. Like lesson 30's `ScriptedModel`, but each reply also reports usage:
input grows with the history (it is resent every call, lesson 12) and output is a fixed 40 tokens.

In [9]:
class PricedScript:
    def __init__(self, script: list, name: str, max_out: int):
        self.script, self.name, self.max_out, self.calls = script, name, max_out, 0

    def __call__(self, messages: list[dict], tools: list[dict]) -> dict:
        self.calls += 1
        reply = self.script[min(self.calls, len(self.script)) - 1]
        reply = reply(messages) if callable(reply) else json.loads(json.dumps(reply))
        for i, call in enumerate(reply.get("tool_calls", [])):
            call["id"] = f"call_{self.calls}_{i}"
        return {"role": "assistant", **reply, "usage": (500 + 150 * len(messages), 40)}

def wander(messages: list[dict]) -> dict:            # lesson 30's model that never finishes
    n = sum(m["role"] == "tool" for m in messages) + 1
    return {"tool_calls": [{"name": "search_notes", "args": {"query": f"comet {n}"}}]}

The wandering model on the big model's prices, with a generous step limit of 50, under three budgets:
no real limit ($1.00, a billion tokens), a 50,000-token budget, and the $0.01 dollar budget.

In [10]:
QUESTION = "How many minutes long is a day on Mars?"
for usd, tok in [(1.00, 10**9), (1.00, 50_000), (0.01, 50_000)]:
    run = run_agent(QUESTION, PricedScript([wander], "gemini-3.8-flash", 1024), RunBudget(usd, tok), max_steps=50)
    print(f"${usd:.2f}, {tok:>10,} tokens: stop={run['stop']:10} steps={run['steps']:>2} "
          f"spent=${run['spent_usd']:.5f} tokens={run['tokens']:,}")

$1.00, 1,000,000,000 tokens: stop=step_limit steps=50 spent=$0.30750 tokens=402,000
$1.00,     50,000 tokens: stop=budget     steps=16 spent=$0.03720 tokens=47,040
$0.01,     50,000 tokens: stop=budget     steps= 5 spent=$0.00544 tokens=6,450


With no real budget only the step limit ended the run. The token budget stopped it at step 16. The $0.01
budget stopped it at step 5, under the limit, and returned its history like every other stop reason.
Token budgets matter because input grows every step: the token count climbs faster than the step count.

### 5. A daily budget per user
A run budget does not stop a user who starts 1,000 runs. A **daily budget** caps each user's spend per
UTC day. Each run then gets the smaller of the per-run limit and what is left today. The clock is a
parameter, as in lesson 30, so we can test midnight without waiting.

In [11]:
from collections import defaultdict
from datetime import date, datetime, timezone

class DailyBudget:
    def __init__(self, max_usd: float, today=lambda: datetime.now(timezone.utc).date()):
        self.max_usd, self.today, self.spent = max_usd, today, defaultdict(float)

    def left(self, user: str) -> float:
        return self.max_usd - self.spent[(user, self.today())]

    def add(self, user: str, usd: float) -> None:
        self.spent[(user, self.today())] += usd

def scout_run(user: str, question: str, model, daily: DailyBudget, run_limit: float = 0.01) -> dict:
    left = daily.left(user)
    budget = RunBudget(min(run_limit, left))
    run = run_agent(question, model, budget)
    daily.add(user, budget.spent_usd)
    if run["stop"] == "budget" and left < run_limit:
        run["stop"] = "daily_budget"                     # the day's money ran out, not the run's
    return run

Asha has $0.03 a day and keeps asking the wandering model. Each run costs at most $0.01, until the day
runs dry. Then the next day starts at zero.

In [12]:
day = {"today": date(2026, 10, 9)}
daily = DailyBudget(0.03, today=lambda: day["today"])
for i in range(7):
    run = scout_run("asha", QUESTION, PricedScript([wander], "gemini-3.8-flash", 1024), daily)
    print(f"run {i + 1}: stop={run['stop']:12} steps={run['steps']} spent=${run['spent_usd']:.5f} "
          f"left today=${daily.left('asha'):.5f}")
day["today"] = date(2026, 10, 10)
print("next day, left:", f"${daily.left('asha'):.2f}")

run 1: stop=budget       steps=5 spent=$0.00544 left today=$0.02456
run 2: stop=budget       steps=5 spent=$0.00544 left today=$0.01912
run 3: stop=budget       steps=5 spent=$0.00544 left today=$0.01369
run 4: stop=budget       steps=5 spent=$0.00544 left today=$0.00825
run 5: stop=daily_budget steps=4 spent=$0.00390 left today=$0.00435
run 6: stop=daily_budget steps=1 spent=$0.00064 left today=$0.00371
run 7: stop=daily_budget steps=0 spent=$0.00000 left today=$0.00371
next day, left: $0.03


Four full runs, then two short ones with what was left, then a seventh with zero steps: no model call at all.
In production `spent` lives in a shared store (a database or Redis), not in a dict in one process.

### 6. Control 3: small model first
Lesson 33 routed requests to handlers; here the router picks a **model**. Easy questions (one lookup, one
sum) go to `gemini-3.5-flash-lite`; questions that need comparing or explaining go to `gemini-3.8-flash`.
Each model keeps its own settings (lesson 53: 3.8-flash refuses the `MINIMAL` thinking level).

In [13]:
SMALL = {"model": "gemini-3.5-flash-lite", "temperature": 1.0, "thinking_level": "MINIMAL", "max_output_tokens": 300}
BIG = {"model": "gemini-3.8-flash", "temperature": 1.0, "thinking_level": "LOW", "max_output_tokens": 1024}
HARD_WORDS = ("compare", "why", "explain", "difference", "plan")

def pick_model(question: str) -> dict:
    q = question.lower()
    return BIG if any(w in q for w in HARD_WORDS) or len(q.split()) > 25 else SMALL

QUESTIONS = ["When did Perseverance land?", "How many minutes long is a day on Mars?",
             "Compare what my notes say about Mars and Venus."]
for q in QUESTIONS:
    print(f"{pick_model(q)['model']:22} <- {q}")

gemini-3.5-flash-lite  <- When did Perseverance land?
gemini-3.5-flash-lite  <- How many minutes long is a day on Mars?
gemini-3.8-flash       <- Compare what my notes say about Mars and Venus.


A keyword router costs nothing to run. Lesson 33 showed a model router is more accurate on free text;
here a miss is cheap, because the next step escalates.

### 7. Escalate when the small model fails
Routing guesses before the run. A **cascade** checks after: if the small model's run ends without an
answer (step limit, empty text), the same question goes to the big model. Both runs share **one**
`RunBudget`, so escalating can never take the question past $0.01.

In [14]:
def routed_run(question: str, make_model, budget: RunBudget) -> dict:
    settings = pick_model(question)
    run = run_agent(question, make_model(settings), budget, max_steps=4)
    if settings is SMALL and run["stop"] in ("step_limit", "end_of_turn") and not run["answer"]:
        first = run["model"]
        run = run_agent(question, make_model(BIG), budget, max_steps=4)
        run["escalated_from"] = first
    return run

Try it with scripted models: the small one wanders, the big one answers in one call.

In [15]:
def fake_models(settings: dict):
    script = [wander] if settings is SMALL else [{"text": "A Mars day is about 1,479 minutes."}]
    return PricedScript(script, settings["model"], settings["max_output_tokens"])

run = routed_run(QUESTION, fake_models, RunBudget(0.01))
print(run["stop"], "| answered by", run["model"], "| escalated from", run.get("escalated_from"),
      f"| total ${run['spent_usd']:.5f}")

end_of_turn | answered by gemini-3.8-flash | escalated from gemini-3.5-flash-lite | total $0.00236


### 8. Real Gemini under the controls
Lesson 53's `GeminiModel`, now reporting `name`, `max_out` and each reply's `usage` (input, plus output
and thinking). First the history converter from lesson 30, unchanged.

In [16]:
NO_AFC = types.AutomaticFunctionCallingConfig(disable=True)
PROMPT = ("You are Scout. Answer questions about the user's notes. Always search the notes first and "
          "answer only from what they say. If the notes do not cover it, say so. Use the calculator for "
          "any arithmetic. Answer in one sentence.")
DECLS = [types.Tool(function_declarations=[types.FunctionDeclaration(
    name=s["name"], description=s["description"], parameters_json_schema=s["parameters"]) for s in SCHEMAS])]

def to_contents(messages: list[dict]) -> list[types.Content]:
    contents = []
    for m in messages:
        if m["role"] == "user":
            contents.append(types.Content(role="user", parts=[types.Part(text=m["text"])]))
        elif m["role"] == "assistant":
            contents.append(m["raw"])                     # unchanged: keeps the thought signature
        else:
            response = {"error": m["content"]} if m["is_error"] else {"output": json.loads(m["content"])}
            part = types.Part(function_response=types.FunctionResponse(id=m["call_id"], name=m["name"], response=response))
            if contents[-1].role == "user" and contents[-1].parts[0].function_response:
                contents[-1].parts.append(part)
            else:
                contents.append(types.Content(role="user", parts=[part]))
    return contents

The wrapper. Settings live next to the model id, as in lesson 53.

In [17]:
class GeminiModel:
    def __init__(self, settings: dict):
        self.settings, self.name, self.max_out = settings, settings["model"], settings["max_output_tokens"]
        self.config = types.GenerateContentConfig(
            system_instruction=PROMPT, temperature=settings["temperature"], max_output_tokens=self.max_out,
            thinking_config=types.ThinkingConfig(thinking_level=settings["thinking_level"]), tools=DECLS,
            automatic_function_calling=NO_AFC)

    def __call__(self, messages: list[dict], tools: list[dict]) -> dict:
        resp = client.models.generate_content(model=self.name, contents=to_contents(messages), config=self.config)
        u = resp.usage_metadata
        usage = (u.prompt_token_count or 0, (u.candidates_token_count or 0) + (u.thoughts_token_count or 0))
        raw = resp.candidates[0].content
        calls = [{"id": fc.id, "name": fc.name, "args": dict(fc.args)} for fc in resp.function_calls or []]
        if calls:
            return {"role": "assistant", "tool_calls": calls, "raw": raw, "usage": usage}
        return {"role": "assistant", "text": resp.text, "raw": raw, "usage": usage}

The three questions twice: routed (small first, $0.01 each), then everything on the big model, as a team
without routing would do. The cell saves the numbers to `../data/recorded/54.json` for the fallback below.

In [18]:
results = {"routed": [], "all_big": []}
for q in QUESTIONS:
    run = routed_run(q, GeminiModel, RunBudget(0.01))
    results["routed"].append({k: run.get(k) for k in ("model", "stop", "steps", "spent_usd", "tokens", "answer")})
    run = run_agent(q, GeminiModel(BIG), RunBudget(0.01))
    results["all_big"].append({k: run.get(k) for k in ("model", "stop", "steps", "spent_usd", "tokens", "answer")})

with open("../data/recorded/54.json", "w") as f:
    json.dump(results, f, indent=1)
for r in results["routed"]:
    print(f"{r['model']:22} {r['stop']:11} steps={r['steps']} ${r['spent_usd']:.5f} | {r['answer']}")

gemini-3.5-flash-lite  end_of_turn steps=2 $0.00021 | The rover Perseverance landed on Mars in February 2021.
gemini-3.5-flash-lite  end_of_turn steps=3 $0.00035 | A day on Mars is 1,479 minutes long.
gemini-3.8-flash       end_of_turn steps=2 $0.00152 | Your notes state that Mars has two small moons (Phobos and Deimos), a day lasting about 24 hours and 39 minutes, and a rover named Perseverance that landed in February 2021, while they note only that Venus spins backwards compared with most planets.


If you skipped the model cell, load the recorded output of that cell (saved on a real run).

In [19]:
if "results" not in globals():   # recorded output of the cell above
    with open("../data/recorded/54.json") as f:
        results = json.load(f)
for mode, runs in results.items():
    total = sum(r["spent_usd"] for r in runs)
    print(f"{mode:8} total ${total:.5f} | per run: " + ", ".join(f"${r['spent_usd']:.5f}" for r in runs)
          + " | stops: " + ", ".join(r["stop"] for r in runs))

routed   total $0.00208 | per run: $0.00021, $0.00035, $0.00152 | stops: end_of_turn, end_of_turn, end_of_turn
all_big  total $0.00309 | per run: $0.00071, $0.00082, $0.00156 | stops: end_of_turn, end_of_turn, end_of_turn


Every run ended with an answer and stayed far under $0.01. Routing made the two easy questions about 2.3 to
3.4 times cheaper; the hard one went to the big model both times, so it cost about the same. Overall that is
$0.0021 against $0.0031, about a third less on this mix. The more easy questions, the more routing pays.

### 9. Control 4: the provider's spend limit
Your budgets only see your own code. The provider enforces its own limits on the whole account:
- **Gemini API**: monthly spend cap per Google Cloud project, shared by all its keys (AI Studio > Spend).
  Marked experimental: billing data lags, so it can be overshot.
- **Anthropic**: the tier's cap returns **429** `enforced_spend_limit_reached` with **no** `retry-after`;
  access is back at 00:00 UTC on the stated date. A limit you set yourself returns **400**.

We build both SDK errors directly: the cap 429 from Anthropic's docs and a normal rate-limit 429.

In [20]:
import anthropic, httpx2

REQUEST = httpx2.Request("POST", "https://api.anthropic.com/v1/messages")
CAP_BODY = {"type": "error", "request_id": "req_stub", "error": {"type": "rate_limit_error",
            "message": "You have reached your API usage limits. You will regain access on 2026-11-01 at 00:00 UTC.",
            "details": {"error_code": "enforced_spend_limit_reached"}}}
RATE_BODY = {"type": "error", "request_id": "req_stub",
             "error": {"type": "rate_limit_error", "message": "Rate limit exceeded: requests per minute."}}

def provider_error(body: dict, headers: dict | None = None) -> anthropic.RateLimitError:
    response = httpx2.Response(429, json=body, headers=headers, request=REQUEST)
    return anthropic.RateLimitError("Error code: 429", response=response, body=body)

print(type(provider_error(CAP_BODY)).__name__, provider_error(CAP_BODY).status_code)

RateLimitError 429


Same class, same status. Only `details.error_code` tells them apart, so Scout checks it and turns the spend
cap into its own stop reason. Below, the rate-limit case with `retry-after=1` is a built example. A rate limit is worth a retry (lesson 14); a spend cap never is.

In [21]:
def is_spend_cap(e: Exception) -> bool:
    details = (getattr(e, "body", None) or {}).get("error", {}).get("details") or {}
    return getattr(e, "status_code", None) == 429 and details.get("error_code") == "enforced_spend_limit_reached"

def provider_stop(e: anthropic.APIStatusError) -> str:
    if is_spend_cap(e):
        return "spend_cap"                       # alert a person; no retry until the date in the message
    return "retry" if e.status_code == 429 or e.status_code >= 500 else "provider_error"

for name, err in [("spend cap", provider_error(CAP_BODY)), ("rate limit", provider_error(RATE_BODY, {"retry-after": "1"}))]:
    print(f"{name:10} -> {provider_stop(err):9} | retry-after={err.response.headers.get('retry-after')} | "
          f"{err.body['error']['message'][:60]}")

spend cap  -> spend_cap | retry-after=None | You have reached your API usage limits. You will regain acce
rate limit -> retry     | retry-after=1 | Rate limit exceeded: requests per minute.


## At work
- **The spend cap is an outage, not a slowdown.** Teams alert on `enforced_spend_limit_reached`, set
  provider limits below the tier cap with headroom, and keep a second provider ready. → lesson 57
- **Provider caps lag, so the app enforces the hard stop.** Gemini's project spend cap is experimental and
  billing data arrives late; batch jobs may run past it. Per-run and per-user budgets in code are the
  real limit. → lesson 55
- **Separate limits per environment and app.** One Anthropic workspace or Google Cloud project per app and
  environment, each with its own spend limit, so a runaway test cannot drain production. → lesson 56
- **Small model first is standard.** Teams route most traffic to the cheap model and escalate on failure,
  then check the escalation rate on a dashboard: if it climbs, routing is costing more than it saves. → lesson 58

## What just happened
- `max_output_tokens` gave each call a worst case: $0.031 at an 8,192 cap on the big model, $0.0009 at 300 on the small one.
- `RunBudget` checked spent + worst case of the next call *before* calling, and refused call 4 at $0.0062 spent.
- Inside the loop, the dollar budget stopped the wandering model at step 5 ($0.0054); a token budget alone let it run 16 steps.
- `DailyBudget` gave Asha four full runs, two short ones, then a zero-step `daily_budget` stop, reset at midnight UTC.
- `pick_model` sent easy questions to flash-lite; `routed_run` escalated failures to 3.8-flash inside the same $0.01.
- On real Gemini, routing cost $0.0021 for the three questions against $0.0031 with the big model only.
- The spend-cap 429 became a `spend_cap` stop; a plain rate-limit 429 with the same class and status became a retry.

## Common mistakes
**1. Checking the budget after the call.** It looks the same, but a run stops only once it is already over.

In [22]:
def run_checking_after(model, budget: RunBudget, max_steps: int = 50) -> RunBudget:
    messages = [{"role": "user", "text": QUESTION}]
    for _ in range(max_steps):
        reply = model(messages, SCHEMAS)
        budget.charge(model.name, *reply.pop("usage"))
        if budget.spent_usd > budget.max_usd:                 # too late: the money is spent
            break
        messages += [reply] + [{"role": "tool", "call_id": c["id"], "name": c["name"], **run_tool(c, TOOLS)}
                               for c in reply.get("tool_calls", [])]
    return budget

late = run_checking_after(PricedScript([wander], "gemini-3.8-flash", 1024), RunBudget(0.01))
assert late.spent_usd <= late.max_usd, f"spent ${late.spent_usd:.5f} of a ${late.max_usd} budget"

AssertionError: spent $0.01140 of a $0.01 budget

Fix: ask `can_afford` with the worst case *before* each call, as `run_agent` does.

In [23]:
fixed = run_agent(QUESTION, PricedScript([wander], "gemini-3.8-flash", 1024), RunBudget(0.01), max_steps=50)
print(f"spent ${fixed['spent_usd']:.5f} of $0.01 | stop={fixed['stop']}")

spent $0.00544 of $0.01 | stop=budget


**2. Routing to a model the price table does not know.** Someone adds a pro model to the router; the
first hard question crashes Scout in the middle of a run.

In [24]:
PRO = {"model": "gemini-3.1-pro-preview", "temperature": 1.0, "thinking_level": "LOW", "max_output_tokens": 1024}
RunBudget(0.01).can_afford(PRO["model"], PRO["max_output_tokens"])

KeyError: 'gemini-3.1-pro-preview'

Fix: check at startup that every model the router can pick has a price, so the error shows on deploy.

In [25]:
def check_prices(*settings: dict) -> None:
    missing = [s["model"] for s in settings if s["model"] not in PRICES_PER_MTOK]
    if missing:
        raise SystemExit(f"no price for {missing}: add them to PRICES_PER_MTOK first")

check_prices(SMALL, BIG)
print("prices ok for", SMALL["model"], "and", BIG["model"])

prices ok for gemini-3.5-flash-lite and gemini-3.8-flash


## Production checklist
- [ ] Every model has a `max_output_tokens` chosen from what its answers need, stored with its id.
- [ ] Every run has a dollar and a token budget, checked before each call with the worst case.
- [ ] Every user (or tenant) has a daily budget in a shared store, keyed by UTC date.
- [ ] Easy requests go to the small model; escalation shares the run's budget; escalation rate is logged.
- [ ] Every routable model has a dated price; checked at startup.
- [ ] Provider spend limits set per app and environment, below the tier cap; alerts on 80% and on the cap.
- [ ] `enforced_spend_limit_reached` is never retried: it becomes a named stop and pages someone, because
  every user is down until the reset date unless traffic moves to another provider.

## Try it
No new model calls needed.
1. Using `results` from step 8, estimate the monthly bill for 1,000 runs a day (30 days) if 80% of
   questions are easy (cost like the first two routed runs, averaged) and 20% are hard (like the third).
   Compare it with sending everything to the big model (average of `results["all_big"]`).
2. The big model's promo price ends on 2026-12-31 (then $1.50 / $7.50). With that price, how many steps does
   the wandering model get under a $0.01 run budget? Change only `PRICES_PER_MTOK`, then put it back.

In [26]:
# your code here

Finally, stop the recorder (in record mode this saves the recording).

In [27]:
llm_replay.stop()

## Key terms
- **run budget** — a limit on the dollars and tokens one agent run may use, checked before each model call with the worst case of that call.
- **daily budget** — a limit on one user's (or tenant's) spend per UTC day; each run gets the smaller of its own limit and what is left today.
- **model routing (small model first)** — sending easy requests to a cheap model and only hard ones to a dearer one, to cut the average cost per run.
- **cascade (escalation)** — retrying a request on a stronger model when the cheaper one fails, inside the same budget.
- **spend limit** — a provider-side cap on an account's spend (Gemini project cap, Anthropic tier or workspace limit); when it is hit every call fails until it resets.